# Notebook 01 — Exploration des données brutes

**Objectif de ce notebook :** avant de nettoyer et fusionner quoi que ce soit, on regarde
chaque fichier tel quel pour comprendre :
- ses dimensions (nombre de lignes/colonnes)
- ses formats de dates (elles sont différentes selon les fichiers !)
- son taux de valeurs manquantes
- les problèmes évidents à corriger dans les notebooks suivants

On ne modifie et on ne sauvegarde **rien** ici — ce notebook est purement exploratoire.

ℹ️ **Format des fichiers : Parquet, pas CSV.** Tout le pipeline (01 à 07) lit et écrit des
fichiers `.parquet` plutôt que `.csv` — plus rapides à charger et plus légers sur disque, et le
type des colonnes (ex: `annee_mois` en texte) est préservé automatiquement d'un notebook à
l'autre, sans avoir besoin de le forcer à chaque lecture. Convertis tes 3 fichiers bruts en
`.parquet` (ex: `pd.read_csv(...).to_parquet(...)`) et dépose-les dans `data/raw/` avant de
lancer ce notebook.

**Fichiers attendus dans `data/raw/`** (renomme tes fichiers complets ainsi si besoin) :
- `datashare.parquet` — les 94 caractéristiques de Gu, Kelly & Xiu (2020)
- `StockReturn.parquet` — les rendements mensuels par entreprise
- `MacroData.parquet` — les variables macroéconomiques (Welch & Goyal)


## 0. Import des bibliothèques

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")  # pour pouvoir importer config.py, situe a la racine du projet
import config

# Affichage un peu plus lisible dans les sorties pandas
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)


## 1. Chargement des 3 fichiers bruts

In [2]:
chars = pd.read_parquet(config.FICHIER_CARACTERISTIQUES_BRUT)
returns = pd.read_parquet(config.FICHIER_RETURNS_BRUT)
macro = pd.read_parquet(config.FICHIER_MACRO_BRUT)

print("Chargement termine.")


Chargement termine.


## 2. Premier coup d'oeil — dimensions et colonnes

In [3]:
for nom, df in [("Caracteristiques (datashare)", chars),
                ("Rendements (StockReturn)", returns),
                ("Macro (MacroData)", macro)]:
    print(f"--- {nom} ---")
    print(f"Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    print(f"Colonnes   : {list(df.columns)[:8]}{' ...' if df.shape[1] > 8 else ''}")
    print()


--- Caracteristiques (datashare) ---
Dimensions : 4117300 lignes x 97 colonnes
Colonnes   : ['permno', 'DATE', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol'] ...

--- Rendements (StockReturn) ---
Dimensions : 4859751 lignes x 3 colonnes
Colonnes   : ['PERMNO', 'date', 'RET']

--- Macro (MacroData) ---
Dimensions : 1860 lignes x 18 colonnes
Colonnes   : ['yyyymm', 'Index', 'D12', 'E12', 'b/m', 'tbl', 'AAA', 'BAA'] ...



In [4]:
chars.head()

,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,stdcf,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,sic2
0,10006,19570131,82249.000,1.122846,1.260784,0.047180,9.569953,0.025742,0.046433,0.044843,...,NaN,NaN,0.013234,9.411565e-08,0.015453,0.008058,0.355638,0.460420,1.120996e-07,37.0
1,10014,19570131,3903.375,0.426734,0.182102,-0.275641,6.237836,0.072103,0.046433,-0.086957,...,NaN,NaN,0.033305,6.610609e-06,0.047619,0.033495,1.152126,1.169610,9.229146e-08,NaN
2,10022,19570131,9273.250,1.066449,1.137313,-0.025490,7.008844,0.027648,0.046433,-0.060377,...,NaN,NaN,0.016023,2.286832e-06,0.020833,0.015589,0.815777,0.679803,1.181757e-07,NaN
3,10030,19570131,54465.875,0.926038,0.857547,0.018171,9.825337,0.021700,0.046433,0.044633,...,NaN,NaN,0.015295,1.464273e-07,0.039326,0.015849,0.739302,1.333656,6.126699e-08,NaN
4,10057,19570131,40250.000,1.247748,1.556875,0.025785,7.901007,0.025506,0.046433,0.086667,...,NaN,NaN,0.005954,1.380375e-06,0.056856,0.019945,0.755510,0.410391,3.315790e+00,NaN


In [5]:
returns.head()

,PERMNO,date,RET
0,10000,19851231,NaN
1,10000,19860131,C
2,10000,19860228,-0.257143
3,10000,19860331,0.365385
4,10000,19860430,-0.098592


In [6]:
macro.head()

,yyyymm,Index,D12,E12,b/m,tbl,AAA,BAA,lty,ntis,Rfree,infl,ltr,corpr,svar,csp,CRSP_SPvw,CRSP_SPvwx
0,187101,4.44,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,187102,4.50,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,187103,4.61,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004525,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,187104,4.74,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004252,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,187105,4.86,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004643,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Caractéristiques (`datashare.parquet`) — exploration détaillée

Cette base contient une ligne par (entreprise, mois). La colonne `permno` identifie
l'entreprise, `DATE` est au format `AAAAMMJJ` (ex: 19570131 = janvier 1957), et
`sic2` donne le secteur d'activité (déjà inclus, pas besoin d'une base séparée).

In [7]:
print("Types de donnees :")
print(chars.dtypes.value_counts())
print()
print("Nombre d'entreprises distinctes (permno) :", chars['permno'].nunique())
print("Periode couverte (colonne DATE) : de", chars['DATE'].min(), "a", chars['DATE'].max())


Types de donnees :
float64    95
int64       2
Name: count, dtype: int64

Nombre d'entreprises distinctes (permno) : 32793
Periode couverte (colonne DATE) : de 19570131 a 20211231


In [8]:
# Pourcentage de valeurs manquantes par colonne, triees de la pire a la meilleure
missing_chars = chars.isna().mean().sort_values(ascending=False) * 100
missing_chars.round(1).head(20)


realestate         73.7
rd_sale            65.0
rd_mve             64.4
stdacc             60.9
stdcf              60.9
secured            60.8
roavol             49.4
grltnoa            48.4
pchsaleinv         47.5
orgcap             47.4
pchsale_pchinvt    46.9
cinvest            44.8
chtx               44.0
pchsale_pchxsga    44.0
saleinv            42.5
cash               41.8
roaq               41.3
ms                 40.9
rsup               40.8
grcapx             40.4
dtype: float64

**A retenir :** dans la vraie base GKX, le taux de valeurs manquantes est très élevé
au début de l'échantillon (années 1950-60) car beaucoup de caractéristiques n'étaient pas
calculables faute de données comptables disponibles. C'est une des raisons pour lesquelles
on envisageait de démarrer l'échantillon plus tard (ex: 1980 ou 1990) plutôt qu'en 1957.

In [9]:
# Secteur (sic2) : distribution et taux de manquants
print("Valeurs manquantes pour sic2 :", chars['sic2'].isna().mean().round(3) * 100, "%")
chars['sic2'].value_counts(dropna=True).head(10)


Valeurs manquantes pour sic2 : 7.3 %


sic2
67.0    404882
60.0    308809
73.0    283834
28.0    259365
36.0    227578
35.0    188978
38.0    175685
49.0    141660
13.0    135588
48.0     97550
Name: count, dtype: int64

## 4. Rendements (`StockReturn.parquet`) — exploration détaillée

Attention : la colonne `RET` de CRSP mélange des nombres (rendements) et des **codes texte**
(ex: `"C"`) qui signifient une donnée manquante ou un delisting. Il faut les repérer avant
de pouvoir faire le moindre calcul dessus.

In [10]:
print("Dimensions :", returns.shape)
print("Type de la colonne RET :", returns['RET'].dtype)
print("Periode couverte : de", returns['date'].min(), "a", returns['date'].max())
print("Nombre d'entreprises distinctes (PERMNO) :", returns['PERMNO'].nunique())


Dimensions : (4859751, 3)
Type de la colonne RET : str
Periode couverte : de 19560131 a 20241231
Nombre d'entreprises distinctes (PERMNO) : 38472


In [11]:
# On essaie de convertir RET en nombre ; tout ce qui n'est pas un nombre devient NaN
ret_numerique = pd.to_numeric(returns['RET'], errors='coerce')

# Les valeurs qui existaient (non vides) mais qui ne sont PAS des nombres = codes CRSP a traiter
valeurs_non_numeriques = returns.loc[returns['RET'].notna() & ret_numerique.isna(), 'RET']
print("Valeurs texte trouvees dans RET (codes CRSP a traiter au notebook 03) :")
print(valeurs_non_numeriques.value_counts())


Valeurs texte trouvees dans RET (codes CRSP a traiter au notebook 03) :
RET
B    65232
C    39204
Name: count, dtype: int64


In [12]:
print(f"% de RET manquant ou non numerique : {ret_numerique.isna().mean() * 100:.1f} %")


% de RET manquant ou non numerique : 3.9 %


## 5. Variables macro (`MacroData.parquet`) — exploration détaillée

Cette base est au format Welch & Goyal : une ligne par mois (`yyyymm`), sans identifiant
d'entreprise. Elle démarre en 1871, bien avant les deux autres fichiers — il faudra la
restreindre à la période utile lors de la fusion.

In [13]:
print("Dimensions :", macro.shape)
print("Periode couverte (yyyymm) : de", macro['yyyymm'].min(), "a", macro['yyyymm'].max())


Dimensions : (1860, 18)
Periode couverte (yyyymm) : de 187101 a 202512


In [14]:
missing_macro = macro.isna().mean().sort_values(ascending=False) * 100
missing_macro.round(1)


csp           57.6
ntis          36.1
ltr           35.5
CRSP_SPvwx    35.5
corpr         35.5
CRSP_SPvw     35.5
b/m           32.4
tbl           31.6
AAA           31.0
BAA           31.0
lty           31.0
infl          27.2
svar           9.1
Rfree          0.1
Index          0.0
yyyymm         0.0
E12            0.0
D12            0.0
dtype: float64

**A retenir :** certaines colonnes macro (comme `tbl`, `AAA`, `BAA`) sont vides sur
les premières décennies car ces séries n'existaient pas encore à l'époque. Il faudra
vérifier, une fois restreint à la période utile pour ton projet (ex: à partir de 1980),
si ces colonnes sont bien remplies sur cette période-là.

## 6. Vérification des clés de fusion (dates & identifiants)

C'est l'étape la plus importante de cette exploration : `datashare` et `StockReturn`
utilisent un format `AAAAMMJJ`, alors que `MacroData` utilise `AAAAMM`. Il faut créer
une clé "année-mois" commune aux 3 fichiers avant de pouvoir les fusionner (ce sera fait
au notebook 03, partie A).

In [15]:
# On cree une colonne "annee_mois" (format AAAAMM, texte) dans chaque fichier
chars['annee_mois'] = chars['DATE'].astype(str).str[:6]
returns['annee_mois'] = returns['date'].astype(str).str[:6]
macro['annee_mois'] = macro['yyyymm'].astype(str)

print(chars[['DATE', 'annee_mois']].head(3))
print(returns[['date', 'annee_mois']].head(3))
print(macro[['yyyymm', 'annee_mois']].head(3))


       DATE annee_mois
0  19570131     195701
1  19570131     195701
2  19570131     195701
       date annee_mois
0  19851231     198512
1  19860131     198601
2  19860228     198602
   yyyymm annee_mois
0  187101     187101
1  187102     187102
2  187103     187103


In [16]:
# Chevauchement des entreprises entre datashare et StockReturn
permnos_chars = set(chars['permno'].unique())
permnos_returns = set(returns['PERMNO'].unique())
communs = permnos_chars & permnos_returns

print(f"Entreprises dans datashare   : {len(permnos_chars)}")
print(f"Entreprises dans StockReturn : {len(permnos_returns)}")
print(f"Entreprises communes aux deux : {len(communs)}")
print("(Sur un simple echantillon de 10 lignes, ce chevauchement peut etre nul ou tres faible : normal.)")


Entreprises dans datashare   : 32793
Entreprises dans StockReturn : 38472
Entreprises communes aux deux : 32791
(Sur un simple echantillon de 10 lignes, ce chevauchement peut etre nul ou tres faible : normal.)


In [17]:
# Chevauchement des periodes (annee_mois) entre les 3 fichiers
periodes_chars = set(chars['annee_mois'])
periodes_returns = set(returns['annee_mois'])
periodes_macro = set(macro['annee_mois'])

print("Periodes datashare   :", sorted(periodes_chars)[:5], "...")
print("Periodes StockReturn :", sorted(periodes_returns)[:5], "...")
print("Periodes MacroData   :", sorted(periodes_macro)[:5], "...")


Periodes datashare   : ['195701', '195702', '195703', '195704', '195705'] ...
Periodes StockReturn : ['195601', '195602', '195603', '195604', '195605'] ...
Periodes MacroData   : ['187101', '187102', '187103', '187104', '187105'] ...


## 7. Résumé et prochaines étapes

À partir de ce que ce notebook a révélé, voici ce qu'il faudra traiter dans les
notebooks suivants (à compléter/ajuster une fois lancé sur les fichiers complets) :

- [ ] **Notebook 02, partie A (caractéristiques)** : décider de la période de départ (1957 vs
  1980/1990) compte tenu du taux de valeurs manquantes ; filtrer automatiquement les 94
  candidates trop incomplètes sur cette période (section A.3bis, seuil
  `config.SEUIL_MAX_PCT_MANQUANT_CARACTERISTIQUES`) ; traiter `sic2` (secteur).
- [ ] **Notebook 02, partie B (rendements)** : convertir `RET` en numérique avec
  `pd.to_numeric(..., errors='coerce')` et documenter les codes CRSP rencontrés (ex: `"C"`)
  avant de les transformer en `NaN`.
- [ ] **Notebook 02, partie C (macro)** : restreindre `MacroData` à la période utile du projet
  et vérifier qu'il ne reste pas de colonnes vides sur cette période.
- [ ] **Notebook 03, partie A (fusion)** : fusionner les 3 fichiers sur la clé commune
  `annee_mois` (+ `permno`/`PERMNO` pour joindre `chars` et `returns` entre eux), puis joindre
  `macro` sur `annee_mois` seul (une ligne macro s'applique à toutes les entreprises du même
  mois).

**Rappel :** remplace les fichiers d'échantillon dans `data/raw/` par tes fichiers complets
(`datashare.parquet`, `StockReturn.parquet`, `MacroData.parquet`) avant de relancer ce notebook
pour de vrais résultats.
